In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from langchain_chroma import Chroma
except Exception:
    try:
        from langchain_community.vectorstores import Chroma  # type: ignore
    except Exception:
        Chroma = None
        print("Chroma is not installed. Install langchain-chroma or langchain-community to build the vector index.")

try:
    from langchain_community.embeddings import SentenceTransformerEmbeddings
except Exception:
    try:
        from langchain.embeddings import SentenceTransformerEmbeddings  # type: ignore
    except Exception:
        SentenceTransformerEmbeddings = None
        print("SentenceTransformerEmbeddings is not installed. Install sentence-transformers/langchain embedding support to build the vector index.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

@dataclass
class VectorConfig:
    vector_records_path: Path = ROOT / "data" / "processed_reports" / "semantic_vector_records.parquet"
    chroma_dir: Path = ROOT / "storage" / "chroma_semantic"
    chroma_collection: str = "insightviewer_semantic"
    embed_model: str = os.getenv("EMBED_MODEL", "all-MiniLM-L12-v2")

CFG = VectorConfig()
CFG.chroma_dir.mkdir(parents=True, exist_ok=True)
print(f"Chroma target: {CFG.chroma_dir}")

## **Load Vector Records**

In [ ]:
def read_dataframe(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix == ".parquet":
            return pd.read_parquet(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)

    csv_fallback = path.with_suffix(".csv")
    if csv_fallback.exists():
        return pd.read_csv(csv_fallback)

    print(f"Missing artifact: {path}")
    return pd.DataFrame()


vector_records_df = read_dataframe(CFG.vector_records_path)
print("semantic vector records:", vector_records_df.shape)
vector_records_df.head(10)

## **Build Chroma Index**

In [ ]:
BUILD_CHROMA_INDEX = False


def metadata_from_json(value: str) -> dict[str, Any]:
    try:
        meta = json.loads(value)
    except Exception:
        return {}
    return {key: val for key, val in meta.items() if isinstance(val, (str, int, float, bool)) and val is not None}


def build_chroma_index(cfg: VectorConfig, records: pd.DataFrame) -> int:
    if records.empty:
        print("No vector records to index.")
        return 0
    if Chroma is None or SentenceTransformerEmbeddings is None:
        print("Skipping Chroma index because Chroma or embeddings support is not installed.")
        return 0

    embeddings = SentenceTransformerEmbeddings(model_name=cfg.embed_model)
    store = Chroma(
        collection_name=cfg.chroma_collection,
        persist_directory=str(cfg.chroma_dir),
        embedding_function=embeddings,
    )

    texts = records["document"].astype(str).tolist()
    ids = records["vector_id"].astype(str).tolist()
    metadatas = [metadata_from_json(value) for value in records["metadata_json"].astype(str).tolist()]
    store.add_texts(texts=texts, metadatas=metadatas, ids=ids)
    try:
        store.persist()  # type: ignore[attr-defined]
    except Exception:
        pass

    print(f"Indexed {len(texts)} records into {cfg.chroma_dir}")
    return len(texts)


if BUILD_CHROMA_INDEX:
    build_chroma_index(CFG, vector_records_df)
else:
    print("Chroma indexing is disabled. Set BUILD_CHROMA_INDEX = True to create embeddings.")

## **Search Example**

In [ ]:
def semantic_search(query: str, k: int = 5) -> list[dict[str, Any]]:
    if Chroma is None or SentenceTransformerEmbeddings is None:
        print("Chroma search is unavailable. Build the index after installing Chroma and embeddings support.")
        return []

    embeddings = SentenceTransformerEmbeddings(model_name=CFG.embed_model)
    store = Chroma(
        collection_name=CFG.chroma_collection,
        persist_directory=str(CFG.chroma_dir),
        embedding_function=embeddings,
    )
    results = store.similarity_search_with_score(query, k=k)
    return [
        {
            "rank": rank,
            "score": float(score),
            "preview": doc.page_content[:500],
            "metadata": doc.metadata,
        }
        for rank, (doc, score) in enumerate(results, start=1)
    ]


# semantic_search("What drove revenue growth for Microsoft?", k=5)